## Building Kepler dashboard for socioeconomic data

This notebook is dedicated for testing the Kepler.gl dashboard for socioeconomic DE pipeline project.

In [6]:
import boto3
import awswrangler as wr
import os

In [7]:
os.environ['AWS_PROFILE'] = 'my-dev-profile'

In [8]:
session = boto3.Session(region_name='us-east-1')

df = wr.athena.read_sql_query(
    sql="""
        SELECT
            geography_id,
            geography_name,
            total_population,
            median_household_income,
            poverty_rate,
            bachelors_plus_rate,
            renter_rate,
            remote_work_rate,
            geometry_wkt
        FROM mart_socioeconomic_tracts
        WHERE state_fips = '06'
        AND survey_year = 2024
    """,
    database="population_demographics_gold_marts",
    workgroup="population-demographics",
    boto3_session=session
)

print(f"Loaded {len(df)} rows")
print(df.head(2))

Loaded 9129 rows
  geography_id                                   geography_name  \
0  06107004104    Census Tract 41.04; Tulare County; California   
1  06109003103  Census Tract 31.03; Tuolumne County; California   

   total_population  median_household_income  poverty_rate  \
0              5234                    48512         31.22   
1               819                    62059         11.89   

   bachelors_plus_rate  renter_rate  remote_work_rate  \
0                 6.49        79.88              2.63   
1                26.77        10.09              2.96   

                                        geometry_wkt  
0  POLYGON ((-119.026291 36.051677, -119.026268 3...  
1  POLYGON ((-120.234155 38.005934, -120.234132 3...  


Let's try to render some Polygons

In [ ]:
import geopandas as gpd
from shapely import wkt
import numpy as np
from lonboard import Map, PolygonLayer
from lonboard.colormap import apply_continuous_cmap
import matplotlib.cm as cm

In [ ]:
# Convert WKT to GeoDataFrame
df['geometry'] = df['geometry_wkt'].apply(wkt.loads)
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')

# Normalize median_household_income for color mapping
values = gdf['median_household_income'].fillna(0).values.astype(float)
normalized = (values - values.min()) / (values.max() - values.min())

# Apply colormap - use matplotlib colormap object not string
colors = apply_continuous_cmap(normalized, cm.viridis, alpha=200)

# Create polygon layer
layer = PolygonLayer.from_geopandas(
    gdf,
    get_fill_color=colors,
    get_line_color=[255, 255, 255, 50],
    get_line_width=10,
    pickable=True,
)

m = Map(layer)
# Save map as HTML file and open in browser
m.to_html("ca_socioeconomic_map.html")